# Cat Breed Classification with PyTorch - Improved Model

This notebook implements an **improved** cat breed classification model using PyTorch and transfer learning with ResNet-50. It addresses class imbalance and uses advanced training techniques to achieve higher accuracy.

**Features:**
- Transfer learning with ResNet-50 fine-tuning
- Class-weighted loss for imbalanced dataset
- Enhanced data augmentation
- Advanced optimization (AdamW + Cosine Annealing)
- Early stopping to prevent overfitting
- GPU acceleration on Google Colab
- Comprehensive evaluation and visualization
- Kaggle dataset integration

**Dataset:** Kaggle Cat Breeds Dataset (67 breeds, ~126K images with severe class imbalance)

## 1. Setup Google Colab Environment

In [ ]:
# Check if GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. Training will be slow.")

# Enable GPU usage
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install numpy pandas matplotlib seaborn scikit-learn pillow tqdm kagglehub

## 3. Download Dataset from Kaggle

In [ ]:
# Use Colab cache for faster access (no download needed!)
import os

# Check if dataset is already in Colab cache
CACHE_DIR = "/root/.cache/kagglehub/datasets/ma7555/cat-breeds-dataset/versions/1"
DATASET_ROOT = CACHE_DIR

if os.path.exists(CACHE_DIR):
    print("✅ Dataset found in Colab cache!")
    print(f"Path: {CACHE_DIR}")

    # Set paths
    IMAGES_PATH = os.path.join(DATASET_ROOT, "images")
    CSV_PATH = os.path.join(DATASET_ROOT, "data", "cats.csv")

    print(f"Dataset root set to: {DATASET_ROOT}")
    print(f"Images path: {IMAGES_PATH}")

    # Quick check
    !ls -la "$DATASET_ROOT" | head -10
    print("... (dataset ready to use)")

else:
    print("📥 Dataset not in cache, downloading...")
    import kagglehub

    path = kagglehub.dataset_download("ma7555/cat-breeds-dataset")
    print("Path to dataset files:", path)

    # Use the downloaded path
    DATASET_ROOT = path
    IMAGES_PATH = os.path.join(DATASET_ROOT, "images")
    CSV_PATH = os.path.join(DATASET_ROOT, "data", "cats.csv")

    print(f"Dataset root set to: {DATASET_ROOT}")
    print(f"Images path: {IMAGES_PATH}")

    # Check downloaded files
    !ls -la "$DATASET_ROOT"

## 4. Import Libraries and Configuration

In [ ]:
# Import all required libraries
import os
import random
import copy
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import time

# Set random seeds for reproducibility
def seed_torch(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_torch()

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Configuration parameters
# DATASET_ROOT, IMAGES_PATH, CSV_PATH are now set in the download cell above
# to use the correct kagglehub path

# Training parameters - increased for better convergence
BATCH_SIZE = 32
NUM_EPOCHS = 25  # Increased from 15
LEARNING_RATE = 0.001
NUM_WORKERS = 2  # Use more workers on Colab

# Model parameters
NUM_CLASSES = 67
MODEL_SAVE_PATH = "/content/cat_breed_classifier.pth"
FINAL_MODEL_PATH = "/content/cat_breed_classifier_final.pth"

# Image parameters
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Data split
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2

# Random seed
RANDOM_SEED = 42

# Device (GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Configuration loaded:")
print(f"Dataset: {DATASET_ROOT}")
print(f"Device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS} (increased for better training)")
print(f"Learning rate: {LEARNING_RATE}")

## 5. Data Loading and Preprocessing

In [ ]:
def get_data_transforms():
    """Get data transformations for training and validation with enhanced augmentation"""
    normalize = transforms.Normalize(mean=MEAN, std=STD)

    train_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(IMAGE_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
        transforms.RandomErasing(p=0.1),
        transforms.ToTensor(),
        normalize
    ])

    val_transforms = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        normalize
    ])

    return train_transforms, val_transforms

def is_image_valid(image_path):
    """Check if an image file is valid and can be opened and converted"""
    try:
        with Image.open(image_path) as img:
            img.load()
            img.convert("RGB")
        return True
    except (IOError, OSError, Image.DecompressionBombError, ValueError, TypeError):
        return False

class ValidImageDataset(Dataset):
    """Custom dataset class that filters out corrupted images"""

    def __init__(self, samples, classes, transform=None):
        self.samples = samples
        self.transform = transform
        self.classes = classes

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Warning: Failed to load image {img_path}: {e}")
            if self.transform:
                placeholder = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color='black')
                return self.transform(placeholder), label
            else:
                placeholder = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color='black')
                return placeholder, label

def load_and_filter_dataset():
    """Load dataset, validate images, and filter out corrupted ones"""
    print("Loading and validating dataset...")

    seed_torch(RANDOM_SEED)

    if not os.path.exists(DATASET_ROOT):
        raise FileNotFoundError(f"Dataset not found at {DATASET_ROOT}")

    dataset_path = IMAGES_PATH if os.path.exists(IMAGES_PATH) else DATASET_ROOT
    print(f"Using dataset path: {dataset_path}")

    train_transforms, val_transforms = get_data_transforms()

    print("Validating images in dataset...")
    valid_samples = []

    temp_dataset = datasets.ImageFolder(root=dataset_path, transform=None)

    for class_idx, class_name in enumerate(temp_dataset.classes):
        class_path = os.path.join(dataset_path, class_name)
        if os.path.isdir(class_path):
            image_files = [f for f in os.listdir(class_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            valid_count = 0
            for img_file in image_files:
                img_path = os.path.join(class_path, img_file)
                if is_image_valid(img_path):
                    valid_samples.append((img_path, class_idx))
                    valid_count += 1
                else:
                    print(f"Skipping corrupted image: {img_path}")
            print(f"Class '{class_name}': {valid_count}/{len(image_files)} valid images")

    print(f"\nTotal valid images: {len(valid_samples)}")
    print(f"Original dataset size: {len(temp_dataset)}")

    full_dataset = ValidImageDataset(valid_samples, temp_dataset.classes, transform=None)

    train_size = int(TRAIN_SPLIT * len(full_dataset))
    val_size = len(full_dataset) - train_size

    train_dataset, val_dataset = random_split(
        full_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(RANDOM_SEED)
    )

    train_dataset.dataset.transform = train_transforms
    val_dataset.dataset.transform = val_transforms

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )

    global class_names, class_to_idx, idx_to_class
    class_names = full_dataset.classes
    class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    print(f"Dataset loaded successfully!")
    print(f"Classes: {len(class_names)}")
    print(f"Training images: {len(train_dataset)}")
    print(f"Validation images: {len(val_dataset)}")

    return train_loader, val_loader, full_dataset, train_dataset, val_dataset

# Global variables
class_names = None
class_to_idx = None
idx_to_class = None

## 6. Model Definition

In [ ]:
def save_model(model, optimizer, scheduler, class_names, history, save_path):
    """Save model checkpoint"""
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scheduler_type': type(scheduler).__name__,
        'class_names': class_names,
        'num_classes': len(class_names),
        'history': history
    }
    torch.save(checkpoint, save_path)
    print(f"Model saved to: {save_path}")

def load_model(model_path, device=DEVICE):
    """Load model checkpoint"""
    checkpoint = torch.load(model_path, map_location=device)

    model = create_model(checkpoint['num_classes'])
    model.load_state_dict(checkpoint['model_state_dict'])

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Recreate scheduler based on type
    scheduler_type = checkpoint.get('scheduler_type', 'StepLR')
    if scheduler_type == 'CosineAnnealingLR':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    else:
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    # Try to load scheduler state if available
    if 'scheduler_state_dict' in checkpoint:
        try:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        except:
            print("Could not load scheduler state, using default")

    class_names = checkpoint['class_names']
    history = checkpoint.get('history', {})

    print(f"Model loaded from: {model_path}")
    return model, optimizer, scheduler, class_names, history

In [ ]:
def create_model(num_classes=None):
    """Create ResNet-50 model with transfer learning"""
    if num_classes is None:
        num_classes = NUM_CLASSES

    # Load pretrained ResNet-50 model
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # Freeze all layers initially
    for param in model.parameters():
        param.requires_grad = False

    # Get the number of input features for the final fully connected layer
    num_ftrs = model.fc.in_features
    print(f"ResNet-50 fc input features: {num_ftrs}")

    # Replace the final layer for our classification task
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
        nn.LogSoftmax(dim=1)
    )

    # Move model to device
    model = model.to(DEVICE)

    print(f"Model created with {num_classes} output classes")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    return model

def get_criterion_optimizer_scheduler(model, class_counts=None):
    """Get loss function, optimizer, and scheduler with optional class weights"""
    if class_counts is not None:
        # Calculate class weights for imbalanced dataset
        total_samples = sum(class_counts)
        class_weights = [total_samples / (len(class_counts) * count) for count in class_counts]
        class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
        criterion = nn.NLLLoss(weight=class_weights)
        print(f"Using weighted loss with class weights computed for {len(class_counts)} classes")
    else:
        criterion = nn.NLLLoss()

    # Use AdamW optimizer with weight decay for better regularization
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    # Use cosine annealing scheduler for better convergence
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    return criterion, optimizer, scheduler

def save_model(model, optimizer, scheduler, class_names, history, save_path):
    """Save model checkpoint"""
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scheduler_type': type(scheduler).__name__,
        'class_names': class_names,
        'num_classes': len(class_names),
        'history': history
    }
    torch.save(checkpoint, save_path)
    print(f"Model saved to: {save_path}")

def load_model(model_path, device=DEVICE):
    """Load model checkpoint"""
    checkpoint = torch.load(model_path, map_location=device)

    model = create_model(checkpoint['num_classes'])
    model.load_state_dict(checkpoint['model_state_dict'])

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Recreate scheduler based on type
    scheduler_type = checkpoint.get('scheduler_type', 'StepLR')
    if scheduler_type == 'CosineAnnealingLR':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    else:
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    # Try to load scheduler state if available
    if 'scheduler_state_dict' in checkpoint:
        try:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        except:
            print("Could not load scheduler state, using default")

    class_names = checkpoint['class_names']
    history = checkpoint.get('history', {})

    print(f"Model loaded from: {model_path}")
    return model, optimizer, scheduler, class_names, history

In [ ]:
def evaluate_model(model, test_loader, class_names, device):
    """Evaluate model on test set and generate detailed metrics"""
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        pbar = tqdm(test_loader, desc='Evaluating', leave=False)
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    print(f"\n{'='*50}")
    print("MODEL EVALUATION RESULTS")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"{'='*50}")

    # Classification report
    print("\nCLASSIFICATION REPORT:")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(20, 16))
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('/content/confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Top 10 most confused breeds
    print("\nTOP 10 MOST CONFUSED BREED PAIRS:")
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    np.fill_diagonal(cm_normalized, 0)  # Remove diagonal

    max_indices = np.unravel_index(np.argsort(cm_normalized.ravel())[-10:], cm_normalized.shape)
    for i in range(len(max_indices[0])):
        true_idx = max_indices[0][i]
        pred_idx = max_indices[1][i]
        confusion_rate = cm_normalized[true_idx, pred_idx]
        if confusion_rate > 0.01:  # Only show significant confusions
            print(f"{class_names[true_idx]} → {class_names[pred_idx]}: {confusion_rate:.3f}")

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }

def analyze_class_performance(results, class_names):
    """Analyze performance per class"""
    cm = results['confusion_matrix']

    class_accuracies = []
    class_precisions = []
    class_recalls = []
    class_f1s = []

    for i in range(len(class_names)):
        # Per-class accuracy (diagonal / row sum)
        class_acc = cm[i, i] / cm[i, :].sum() if cm[i, :].sum() > 0 else 0
        class_accuracies.append(class_acc)

        # Per-class precision (diagonal / column sum)
        class_prec = cm[i, i] / cm[:, i].sum() if cm[:, i].sum() > 0 else 0
        class_precisions.append(class_prec)

        # Per-class recall (same as accuracy for multiclass)
        class_recalls.append(class_acc)

        # Per-class F1
        if class_prec + class_acc > 0:
            class_f1 = 2 * (class_prec * class_acc) / (class_prec + class_acc)
        else:
            class_f1 = 0
        class_f1s.append(class_f1)

    # Create performance dataframe
    perf_df = pd.DataFrame({
        'Breed': class_names,
        'Accuracy': class_accuracies,
        'Precision': class_precisions,
        'Recall': class_recalls,
        'F1-Score': class_f1s,
        'Support': cm.sum(axis=1)
    })

    # Sort by accuracy
    perf_df = perf_df.sort_values('Accuracy', ascending=False)

    print(f"\n{'='*60}")
    print("CLASS PERFORMANCE ANALYSIS")
    print(f"{'='*60}")
    print("Top 10 Best Performing Breeds:")
    print(perf_df.head(10)[['Breed', 'Accuracy', 'Support']].to_string(index=False))

    print("\nBottom 10 Worst Performing Breeds:")
    print(perf_df.tail(10)[['Breed', 'Accuracy', 'Support']].to_string(index=False))

    # Plot class accuracies
    plt.figure(figsize=(15, 8))
    top_20 = perf_df.head(20)
    plt.barh(range(len(top_20)), top_20['Accuracy'])
    plt.yticks(range(len(top_20)), top_20['Breed'])
    plt.xlabel('Accuracy')
    plt.title('Top 20 Breed Classification Accuracies')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/class_accuracies.png', dpi=300, bbox_inches='tight')
    plt.show()

    return perf_df

In [ ]:
def predict_single_image(model, image_path, transform, class_names, device):
    """Predict breed for a single image"""
    model.eval()

    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    original_image = image.copy()

    # Apply transformations
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Make prediction
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]
        predicted_class_idx = torch.argmax(probabilities).item()
        confidence = probabilities[predicted_class_idx].item()

    predicted_breed = class_names[predicted_class_idx]

    # Get top 5 predictions
    top5_prob, top5_idx = torch.topk(probabilities, 5)
    top5_breeds = [class_names[i] for i in top5_idx.cpu().numpy()]
    top5_confidences = top5_prob.cpu().numpy()

    return {
        'predicted_breed': predicted_breed,
        'confidence': confidence,
        'top5_breeds': top5_breeds,
        'top5_confidences': top5_confidences,
        'original_image': original_image
    }

def display_prediction_results(results):
    """Display prediction results with image"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Display image
    ax1.imshow(results['original_image'])
    ax1.set_title(f"Predicted: {results['predicted_breed']}\nConfidence: {results['confidence']:.1%}")
    ax1.axis('off')

    # Display top 5 predictions
    y_pos = np.arange(len(results['top5_breeds']))
    ax2.barh(y_pos, results['top5_confidences'], align='center')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(results['top5_breeds'])
    ax2.invert_yaxis()
    ax2.set_xlabel('Confidence')
    ax2.set_title('Top 5 Predictions')
    ax2.grid(True, alpha=0.3)

    # Add percentage labels on bars
    for i, v in enumerate(results['top5_confidences']):
        ax2.text(v + 0.01, i, f'{v:.1%}', va='center')

    plt.tight_layout()
    plt.show()

    print(f"\nPrediction Results:")
    print(f"Predicted Breed: {results['predicted_breed']}")
    print(f"Confidence: {results['confidence']:.1%}")
    print("\nTop 5 Predictions:")
    for i, (breed, conf) in enumerate(zip(results['top5_breeds'], results['top5_confidences']), 1):
        print(f"{i}. {breed}: {conf:.1%}")

def manual_image_testing(model, transform, class_names, device):
    """Interactive function for testing uploaded images"""
    print("Manual Image Testing")
    print("=" * 50)
    print("Upload an image of a cat to classify its breed.")
    print("Supported formats: JPG, PNG, JPEG")
    print("For best results, use clear photos of single cats.")
    print()

    # Create file upload widget (Colab only)
    try:
        from google.colab import files
        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded.")
            return

        # Get the uploaded file
        file_name = list(uploaded.keys())[0]
        image_path = f'/content/{file_name}'

        # Save uploaded file
        with open(image_path, 'wb') as f:
            f.write(uploaded[file_name])

        print(f"File uploaded: {file_name}")

    except ImportError:
        # Local environment - ask for file path
        print("Running in local environment.")
        image_path = input("Enter the path to your cat image: ").strip()

        if not os.path.exists(image_path):
            print(f"File not found: {image_path}")
            return

    try:
        # Make prediction
        results = predict_single_image(model, image_path, transform, class_names, device)

        # Display results
        display_prediction_results(results)

    except Exception as e:
        print(f"Error processing image: {e}")
        print("Make sure the image is a valid cat photo in JPG/PNG format.")

    finally:
        # Clean up uploaded file
        if 'uploaded' in locals() and uploaded:
            try:
                os.remove(image_path)
                print(f"Cleaned up temporary file: {file_name}")
            except:
                pass

In [ ]:
def save_model(model, optimizer, scheduler, class_names, history, save_path):
    """Save model checkpoint with training state"""
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'class_names': class_names,
        'history': history,
        'num_classes': len(class_names),
        'save_time': time.strftime('%Y-%m-%d %H:%M:%S')
    }

    torch.save(checkpoint, save_path)
    print(f"Model saved to {save_path}")

def load_model(checkpoint_path, device):
    """Load model checkpoint"""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Create model
    num_classes = checkpoint['num_classes']
    model = create_model(num_classes)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    class_names = checkpoint['class_names']
    history = checkpoint.get('history', {})

    print(f"Model loaded from {checkpoint_path}")
    print(f"Number of classes: {num_classes}")
    print(f"Training history available: {'Yes' if history else 'No'}")

    return model, class_names, history

def get_model_size(model):
    """Calculate model size in MB"""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    size_mb = (param_size + buffer_size) / 1024 / 1024
    return size_mb

def print_model_summary(model, class_names):
    """Print model architecture summary"""
    print(f"\n{'='*50}")
    print("MODEL ARCHITECTURE SUMMARY")
    print(f"{'='*50}")
    print(f"Model: ResNet-50 with Custom Classifier")
    print(f"Number of classes: {len(class_names)}")
    print(f"Model size: {get_model_size(model):.2f} MB")

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Print class names sample
    print(f"\nClass names (first 10): {class_names[:10]}")
    if len(class_names) > 10:
        print(f"... and {len(class_names) - 10} more breeds")

    print(f"{'='*50}")

def setup_device():
    """Setup and return the appropriate device"""
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"Using CUDA device: {torch.cuda.get_device_name()}")
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    else:
        device = torch.device('cpu')
        print("Using CPU device")

    return device

def set_seed(seed=42):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Make deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Random seed set to: {seed}")

def check_dataset_balance(labels, class_names):
    """Check and display dataset balance"""
    unique, counts = np.unique(labels, return_counts=True)

    print(f"\n{'='*50}")
    print("DATASET BALANCE ANALYSIS")
    print(f"{'='*50}")
    print(f"Total samples: {len(labels)}")
    print(f"Number of classes: {len(unique)}")

    # Calculate balance metrics
    min_samples = counts.min()
    max_samples = counts.max()
    mean_samples = counts.mean()
    std_samples = counts.std()

    print(f"Samples per class - Min: {min_samples}, Max: {max_samples}, Mean: {mean_samples:.1f}, Std: {std_samples:.1f}")

    # Check for imbalance
    imbalance_ratio = max_samples / min_samples
    print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

    if imbalance_ratio > 3:
        print("⚠️  SEVERE CLASS IMBALANCE DETECTED!")
        print("   Class-weighted loss will be applied during training.")
    elif imbalance_ratio > 2:
        print("⚠️  Moderate class imbalance detected.")
    else:
        print("✅ Dataset appears balanced.")

    # Show top 5 most/least represented breeds
    breed_counts = list(zip(class_names, counts))
    breed_counts.sort(key=lambda x: x[1], reverse=True)

    print(f"\nTop 5 most represented breeds:")
    for breed, count in breed_counts[:5]:
        print(f"  {breed}: {count} samples")

    print(f"\nTop 5 least represented breeds:")
    for breed, count in breed_counts[-5:]:
        print(f"  {breed}: {count} samples")

    print(f"{'='*50}")

    return counts

In [ ]:
# =============================================
# 10. MAIN EXECUTION
# =============================================

# Set random seed for reproducibility
set_seed(42)

# Setup device
DEVICE = setup_device()

# Load and filter dataset
print("Loading dataset...")
train_dataset, val_dataset, test_dataset, class_names = load_and_filter_dataset()

# Check dataset balance
train_labels = [label for _, label in train_dataset.samples]
check_dataset_balance(train_labels, class_names)

# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(train_dataset, val_dataset, test_dataset)

# Create model, criterion, optimizer, scheduler
model = create_model(len(class_names))
criterion, optimizer, scheduler = get_criterion_optimizer_scheduler(model, train_dataset)

# Print model summary
print_model_summary(model, class_names)

# Train the model
MODEL_SAVE_PATH = '/content/cat_breed_classifier_best.pth'
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_path=MODEL_SAVE_PATH
)

# Plot training history
plot_training_history(history)

# Load best model for evaluation
print("\nLoading best model for evaluation...")
best_model, _, _ = load_model(MODEL_SAVE_PATH, DEVICE)

# Evaluate on test set
print("\nEvaluating on test set...")
results = evaluate_model(best_model, test_loader, class_names, DEVICE)

# Analyze class performance
perf_df = analyze_class_performance(results, class_names)

# Save performance analysis
perf_df.to_csv('/content/class_performance_analysis.csv', index=False)
print("Performance analysis saved to: /content/class_performance_analysis.csv")

print(f"\n{'='*60}")
print("TRAINING COMPLETED SUCCESSFULLY!")
print(f"{'='*60}")
print(f"Final Test Accuracy: {results['accuracy']:.1%}")
print(f"Model saved at: {MODEL_SAVE_PATH}")
print(f"Training history plot: /content/training_history.png")
print(f"Confusion matrix: /content/confusion_matrix.png")
print(f"Class accuracies plot: /content/class_accuracies.png")
print(f"Performance analysis: /content/class_performance_analysis.csv")
print(f"{'='*60}")

# =============================================
# 11. MANUAL IMAGE TESTING
# =============================================

# Test with uploaded images
print("\nReady for manual image testing!")
print("Run the next cell to upload and classify your own cat images.")

# Uncomment the line below to run manual testing immediately
# manual_image_testing(best_model, test_transform, class_names, DEVICE)

In [ ]:
# =============================================
# MANUAL IMAGE TESTING CELL
# =============================================
# Run this cell to test the model with your own cat images
# Upload images through the file picker or provide local paths

# Make sure the model is loaded (run the main execution cell first)
try:
    # This will work if the model was trained in this session
    manual_image_testing(best_model, test_transform, class_names, DEVICE)
except NameError:
    # If model not in memory, try to load from saved checkpoint
    try:
        print("Loading saved model for testing...")
        test_model, test_class_names, _ = load_model(MODEL_SAVE_PATH, DEVICE)
        manual_image_testing(test_model, test_transform, test_class_names, DEVICE)
    except Exception as e:
        print(f"Could not load model: {e}")
        print("Please run the main training cell first!")

## 7. Training Functions

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, save_path):
    """Main training function with improved monitoring"""
    print(f"Starting training for {num_epochs} epochs...")
    print(f"Training on device: {DEVICE}")
    print(f"Save path: {save_path}")
    print("-" * 50)

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'learning_rates': []
    }

    best_val_acc = 0.0
    patience = 10  # Early stopping patience
    patience_counter = 0

    start_time = time.time()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, DEVICE)

        # Step the scheduler (cosine annealing)
        scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['learning_rates'].append(current_lr)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")

        # Early stopping check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_model(model, optimizer, scheduler, class_names, history, save_path)
            print(f"New best model saved! (Val Acc: {val_acc:.2f}%)")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break

    total_time = time.time() - start_time
    print(f"\nTraining completed in {total_time:.2f} seconds")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    return history

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc='Training', leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * correct / total:.2f}%'
        })

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

def validate_one_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validating', leave=False)
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct / total:.2f}%'
            })

    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, save_path):
    """Main training function with improved monitoring"""
    print(f"Starting training for {num_epochs} epochs...")
    print(f"Training on device: {DEVICE}")
    print(f"Save path: {save_path}")
    print("-" * 50)

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'learning_rates': []
    }

    best_val_acc = 0.0
    patience = 10  # Early stopping patience
    patience_counter = 0

    start_time = time.time()

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, DEVICE)

        # Step the scheduler (cosine annealing)
        scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['learning_rates'].append(current_lr)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")

        # Early stopping check
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_model(model, optimizer, scheduler, class_names, history, save_path)
            print(f"New best model saved! (Val Acc: {val_acc:.2f}%)")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs")
                break

    total_time = time.time() - start_time
    print(f"\nTraining completed in {total_time:.2f} seconds")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")

    return history

def plot_training_history(history):
    """Plot training history"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    ax3.plot(epochs, history['learning_rates'], 'g-', label='Learning Rate')
    ax3.set_title('Learning Rate Schedule')
    ax3.set_xlabel('Epochs')
    ax3.set_ylabel('Learning Rate')
    ax3.set_yscale('log')
    ax3.legend()
    ax3.grid(True)

    ax4.axis('off')
    final_train_loss = history['train_loss'][-1]
    final_train_acc = history['train_acc'][-1]
    final_val_loss = history['val_loss'][-1]
    final_val_acc = history['val_acc'][-1]

    ax4.text(0.1, 0.8, f'Final Training Loss: {final_train_loss:.4f}', fontsize=12)
    ax4.text(0.1, 0.6, f'Final Training Acc: {final_train_acc:.2f}%', fontsize=12)
    ax4.text(0.1, 0.4, f'Final Validation Loss: {final_val_loss:.4f}', fontsize=12)
    ax4.text(0.1, 0.2, f'Final Validation Acc: {final_val_acc:.2f}%', fontsize=12)

    plt.tight_layout()
    plt.savefig('/content/training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Evaluation Functions

In [ ]:
def evaluate_model(model, test_loader, criterion, device, class_names):
    """Evaluate model on test set"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    print("Evaluating model on test set...")

    with torch.no_grad():
        pbar = tqdm(test_loader, desc='Evaluating')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            probs = torch.exp(outputs)
            _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    test_loss = running_loss / len(test_loader.dataset)
    test_acc = accuracy_score(all_labels, all_preds) * 100

    report = classification_report(all_labels, all_preds, target_names=class_names, labels=list(range(len(class_names))), output_dict=True)

    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")

    return test_loss, test_acc, all_preds, all_labels, all_probs, report

## 9. Load and Prepare Dataset

In [ ]:
# Load the dataset
print("Loading dataset...")
train_loader, val_loader, full_dataset, train_dataset, val_dataset = load_and_filter_dataset()

# Create test loader (using validation set as test for now)
test_loader = val_loader

print(f"\nDataset Summary:")
print(f"- Total classes: {len(class_names)}")
print(f"- Training samples: {len(train_dataset)}")
print(f"- Validation samples: {len(val_dataset)}")
print(f"- Sample breeds: {class_names[:5]}")

## 10. Create and Train Model

In [ ]:
# Create the model with fine-tuning
print("Creating model with fine-tuning...")
model = create_model()

# Unfreeze the last few layers for better fine-tuning
for name, param in model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

print(f"Trainable parameters after unfreezing: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Calculate class counts for weighted loss
class_counts = [0] * len(class_names)
for _, label in full_dataset:
    class_counts[label] += 1

print(f"Class distribution: {class_counts}")

# Get training components with class weights
criterion, optimizer, scheduler = get_criterion_optimizer_scheduler(model, class_counts)

# Increase epochs for better training
NUM_EPOCHS = 25

# Train the model
print("\nStarting training with weighted loss and fine-tuning...")
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=NUM_EPOCHS,
    save_path=MODEL_SAVE_PATH
)

# Plot training history
print("\nPlotting training history...")
plot_training_history(history)

## 11. Evaluate Model

In [ ]:
# Load the best model for evaluation
print("Loading best model for evaluation...")
model, _, _, class_names, _ = load_model(MODEL_SAVE_PATH)

# Evaluate on test set
test_loss, test_acc, all_preds, all_labels, all_probs, report = evaluate_model(
    model, test_loader, criterion, DEVICE, class_names
)

# Plot confusion matrix
plot_confusion_matrix(all_labels, all_preds, class_names)

# Analyze predictions by breed
analyze_predictions_by_breed(report, class_names)

print("\n" + "="*60)
print("FINAL RESULTS:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print("="*60)

## 12. Test Single Image Prediction

In [ ]:
# Test prediction on a sample image
import glob

print("Testing single image prediction...")
sample_images = glob.glob(os.path.join(IMAGES_PATH, "**/*.jpg"), recursive=True)[:5]

for img_path in sample_images:
    try:
        predicted_breed, confidence, top3 = predict_single_image(model, img_path, class_names, DEVICE)

        print(f"\nImage: {os.path.basename(img_path)}")
        print(f"Predicted: {predicted_breed} ({confidence:.2f}%)")
        print("Top 3 predictions:")
        for breed, conf in top3:
            print(f"  {breed}: {conf:.2f}%")
    except Exception as e:
        print(f"Error processing {img_path}: {e}")

## 15. Save Model for Download

In [ ]:
# Save final model
final_save_path = "/content/cat_breed_classifier_final.pth"
save_model(model, optimizer, scheduler, class_names, history, final_save_path)

print(f"Final model saved to: {final_save_path}")

# List files to download
print("\nFiles to download:")
print("- cat_breed_classifier_final.pth (trained model)")
print("- training_history.png (training curves)")
print("- confusion_matrix.png (confusion matrix)")

# Show file sizes
!ls -lh /content/*.pth /content/*.png 2>/dev/null || echo "Some files not found"

## Summary

This notebook successfully implements an improved cat breed classification model with the following enhancements:

### ✅ **Key Improvements Made:**
1. **Class-weighted Loss**: Addresses severe class imbalance with weighted NLLLoss
2. **Fine-tuning**: Unfreezes ResNet-50's layer4 and fc layers for better feature learning
3. **Enhanced Data Augmentation**: Added vertical flip, affine transforms, and random erasing
4. **Better Optimization**: AdamW optimizer with weight decay (1e-4)
5. **Cosine Annealing Scheduler**: Better learning rate scheduling for convergence
6. **Early Stopping**: Prevents overfitting with patience-based stopping
7. **Increased Training**: 25 epochs instead of 15 for better convergence

### 📊 **Expected Results:**
- **Model**: ResNet-50 with fine-tuning and custom classifier
- **Dataset**: 67 cat breeds with severe class imbalance handled via weighting
- **Training**: 25 epochs with advanced augmentation and optimization
- **Target Performance**: Significantly improved accuracy (aiming for 70-80%+)

### 🎯 **Performance Analysis:**
The previous results showed heavy bias toward majority classes (Domestic Short Hair: 10,668 samples).
With weighted loss and fine-tuning, expect:
- Better performance on minority classes
- Reduced bias toward majority classes
- Overall accuracy improvement from ~51% to 70%+

### 🚀 **Next Steps:**
- Download the improved model (`cat_breed_classifier_final.pth`)
- Test on new cat images
- Consider further improvements like focal loss or oversampling
- Deploy in production with confidence thresholds

In [ ]:



# Define paths (ensure these are defined or adjust as needed)
IMAGES_PATH = "/content/data/images"  # Adjust to your dataset path
MODEL_SAVE_PATH = "/content/cat_breed_classifier.pth"  # Adjust to your model path
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# File upload widget
uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',  # Accept only image files
    multiple=False  # Allow only one file
)

# Output widget to display results
output = widgets.Output()

def on_upload_change(change):
    """Handle the file upload event and process the image."""
    with output:
        clear_output()  # Clear previous output
        print("Processing uploaded image...")

        # Check if a file was uploaded
        if not uploader.value:
            print("❌ No file uploaded. Please select an image.")
            return

        try:
            # Get the uploaded file
            uploaded_file = list(uploader.value.values())[0]
            file_name = uploaded_file['metadata']['name']
            file_content = uploaded_file['content']

            # Save the uploaded image temporarily
            temp_image_path = f"/tmp/{file_name}"
            with open(temp_image_path, 'wb') as f:
                f.write(file_content)

            print(f"Image uploaded: {file_name}")

            # Load the trained model (if not already loaded)
            try:
                model
            except NameError:
                print("Loading model...")
                model, _, _, class_names, _ = load_model(MODEL_SAVE_PATH)

            # Make prediction
            predicted_breed, confidence, top3 = predict_single_image(model, temp_image_path, class_names, DEVICE)

            print(f"\n{'='*60}")
            print(f"🖼️  PREDICTION RESULTS")
            print(f"{'='*60}")
            print(f"Image: {file_name}")
            print(f"🎯 Top Prediction: {predicted_breed} ({confidence:.2f}% confidence)")
            print(f"\n📊 Top 3 Predictions:")
            for i, (breed, conf) in enumerate(top3, 1):
                print(f"  {i}. {breed}: {conf:.2f}%")

            # Display the image
            try:
                img = PILImage.open(temp_image_path)
                plt.figure(figsize=(6, 6))
                plt.imshow(img)
                plt.title(f"Predicted: {predicted_breed} ({confidence:.1f}%)", fontsize=14, pad=20)
                plt.axis('off')
                plt.show()
            except Exception as e:
                print(f"Could not display image: {e}")

            # Clean up temporary file
            if os.path.exists(temp_image_path):
                os.remove(temp_image_path)

        except Exception as e:
            print(f"❌ Error processing image: {e}")
            print("Make sure:")
            print("- The image is a valid format (JPG, PNG, etc.)")
            print("- The model is trained and saved at", MODEL_SAVE_PATH)

# Attach the upload handler
uploader.observe(on_upload_change, names='value')

# Display the upload widget
print("Upload an image to test the model:")
display(uploader)
display(output)

# Example usage with dataset images
print(f"\n{'='*60}")
print("💡 QUICK TEST WITH DATASET IMAGE")
print(f"{'='*60}")

try:
    dataset_images = glob.glob(os.path.join(IMAGES_PATH, "**/*.jpg"), recursive=True)
    if dataset_images:
        # Pick a random image from the dataset
        random_image_path = random.choice(dataset_images)
        print(f"Testing with random dataset image: {os.path.basename(random_image_path)}")

        # Load the model if not already loaded
        try:
            model
        except NameError:
            print("Loading model...")
            model, _, _, class_names, _ = load_model(MODEL_SAVE_PATH)

        # Make prediction
        predicted_breed, confidence, top3 = predict_single_image(model, random_image_path, class_names, DEVICE)

        print(f"🎯 Prediction: {predicted_breed} ({confidence:.2f}%)")
        print("Top 3 predictions:")
        for breed, conf in top3:
            print(f"  {breed}: {conf:.2f}%")

        # Display the image
        try:
            img = PILImage.open(random_image_path)
            plt.figure(figsize=(6, 6))  # Fixed syntax error here
            plt.imshow(img)
            plt.title(f"Predicted: {predicted_breed} ({confidence:.1f}%)", fontsize=14, pad=20)
            plt.axis('off')
            plt.show()
        except Exception as e:
            print(f"Could not display dataset image: {e}")
    else:
        print("No dataset images found for testing")
except Exception as e:
    print(f"Could not test with dataset image: {e}")
